In [6]:
%pip install ffsim
%pip install qiskit_ibm_runtime

In [17]:
import math
import numpy as np
import ffsim
import matplotlib.pyplot as plt
import pyscf.cc
import pyscf.mcscf
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.primitives import StatevectorSampler
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [8]:
spin_sq=0

#N2
mol=pyscf.gto.Mole()
mol.build(
    atom=[["N", (0,0,0)], ["N", (1.0, 0, 0)]],
    basis="sto-6g",
    symmetry="Dooh",
)

#define active space
n_frozen=2
active_space=range(n_frozen, mol.nao_nr())


WARN: Unable to to identify input symmetry using original axes.
Different symmetry axes will be used.



In [9]:
#molecular integrals
scf=pyscf.scf.RHF(mol).run()
norb=len(active_space)
n_electrons=int(sum(scf.mo_occ[active_space]))
n_alpha=(n_electrons+mol.spin)//2
n_beta=(n_electrons-mol.spin)//2
nelec=(n_alpha, n_beta)
cas=pyscf.mcscf.CASCI(scf, norb, nelec)
mo=cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy=cas.get_h1cas(mo)
eri=pyscf.ao2mo.restore(1, cas.get_h2cas(mo),norb)

converged SCF energy = -108.464957764796


In [10]:
reference_energy=cas.run().e_tot
print(f"norb={norb}")
print(f"nelec={nelec}")

CASCI E = -108.595987350986  E(CI) = -32.4115475088426  S^2 = 0.0000000
norb=8
nelec=(5, 5)


In [11]:
ccsd=pyscf.cc.CCSD(scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]).run()
t1=ccsd.t1
t2=ccsd.t2

E(CCSD) = -108.5933309085007  E_corr = -0.1283731437052348


In [12]:
import warnings
from qiskit.transpiler import CouplingMap

warnings.formatwarning=lambda msg, *args, **kwargs: f"Warning: {msg}\n"

In [13]:
#ansatz properties
n_reps=1
pairs_aa=[(p, p+1) for p in range(norb-1)]
pairs_ab=None

coupling_map=CouplingMap.from_heavy_hex(3)
backend=GenericBackendV2(
        coupling_map.size(),
        coupling_map=coupling_map,
        basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)

pass_manager, pairs_ab=ffsim.qiskit.generate_lucj_pass_manager(
        backend=backend,
        norb=norb,
        connectivity="heavy-hex",
        interaction_pairs=(pairs_aa, pairs_ab),
        optimization_level=3,
)

ucj_op=ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    optimize=True,
    options=dict(maxiter=1000), #removing this line may improve results
)


Removing interaction (4, 4) from the end.


In [14]:
qubits=QuantumRegister(2*norb, name="q")
circuit=QuantumCircuit(qubits)

circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()

In [15]:
isa_circuit=pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")

Gate counts: OrderedDict({'xx_plus_yy': 86, 'p': 16, 'measure': 16, 'cp': 15, 'x': 10, 'swap': 2, 'barrier': 1})


In [18]:
rng=np.random.default_rng()
sampler=StatevectorSampler(seed=rng)
job=sampler.run([isa_circuit],shots=100_000)

In [19]:
primitive_result=job.result()
pub_result=primitive_result[0]

In [20]:
def is_valid_bitstring(
    bitstring: str, norb:int, nelec: tuple[int, int]
) -> bool:
    n_alpha, n_beta=nelec
    return(
        len(bitstring)== 2*norb
        and bitstring[norb:].count("1")==n_alpha
    )

bit_array=pub_result.data.meas
num_valid=sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction=num_valid/bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")

Fraction of sampled configurations that are valid: 1.0


In [21]:
expected_fraction_random=(
    math.comb(norb, n_alpha)*math.comb(norb, n_beta)/2**(2*norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: "
    f"{expected_fraction_random}"
)

Expected fraction of valid configurations from uniformly random bitstrings: 0.0478515625
